# AgentGuard — DeBERTa Risk Scorer Training

Run on Kaggle with **GPU T4 x2** and **Internet** enabled. Expected runtime: 3–4 hours.

## Setup (one time)

1. Push code + notebook: `.\scripts\push_kaggle_kernel.ps1` (uploads `agentguard-code.zip` dataset and attaches it to this kernel).
2. Open the kernel on Kaggle → **Settings** → enable GPU + Internet → **Run All**.

## Output

Download from the **Output** tab after success:

- `agentguard/models/risk_scorer.onnx`
- `agentguard/models/model.sha256`
- `agentguard/models/tokenizer.json` + `tokenizer_config.json`

In [ ]:
import os
import subprocess
import sys

# Harmless on Kaggle; suppresses noisy TF/XLA plugin registration stderr.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")

# evaluate is not required for training/export; omit it to avoid pip failures on Kaggle.
packages = "transformers datasets optimum[onnxruntime] scikit-learn accelerate sentencepiece protobuf tiktoken"
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        *packages.split(),
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
    ],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(
        "pip install failed — enable Internet in Session options and re-run.\n"
        + result.stderr[-2000:]
    )

if "dependency conflicts" in (result.stderr + result.stdout).lower():
    print("Note: pip reported RAPIDS dependency warnings (cuml/cudf). Safe to ignore.")

print("Dependencies installed.")

In [ ]:
import os
import shutil
import sys
import tarfile
import zipfile
from pathlib import Path

os.chdir("/kaggle/working")
working = Path("/kaggle/working")
input_root = Path("/kaggle/input")

# Make `import training.*` work for every subprocess in later cells.
os.environ["PYTHONPATH"] = str(working)
if str(working) not in sys.path:
    sys.path.insert(0, str(working))

print(
    "Input datasets:", sorted(p.name for p in input_root.iterdir()) if input_root.exists() else []
)


def copy_tree(src: Path, dest: Path) -> None:
    if src.exists():
        shutil.copytree(src, dest, dirs_exist_ok=True)


def extract_archives(root: Path) -> None:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix == ".zip":
            print(f"Extracting {path.relative_to(input_root)}...")
            with zipfile.ZipFile(path) as zf:
                zf.extractall(working)
        elif path.name.endswith((".tar", ".tar.gz", ".tgz")):
            print(f"Extracting {path.relative_to(input_root)}...")
            with tarfile.open(path) as tf:
                tf.extractall(working)


def materialize_from_input() -> None:
    for train_py in input_root.rglob("train.py"):
        if train_py.parent.name != "training":
            continue
        code_root = train_py.parent.parent
        print(f"Copying code tree from {code_root.relative_to(input_root)}...")
        for name in ("training", "agentguard", "benchmarks"):
            copy_tree(code_root / name, working / name)
        return


extract_archives(input_root)
materialize_from_input()

_TRAINING_FALLBACK = (
    "train.py",
    "prepare_dataset.py",
    "export_onnx.py",
    "probe_checkpoint.py",
    "scoring.py",
    "tokenizer_utils.py",
    "__init__.py",
)

if not (working / "training" / "train.py").exists() and (working / "train.py").exists():
    (working / "training").mkdir(exist_ok=True)
    for name in _TRAINING_FALLBACK:
        src = working / name
        if src.exists():
            shutil.move(src, working / "training" / name)

if not (working / "training" / "train.py").exists():
    print("=== debug: /kaggle/input ===")
    for path in sorted(input_root.rglob("*"))[:40]:
        print(" ", path)
    print("=== debug: /kaggle/working ===")
    for path in sorted(working.rglob("*"))[:40]:
        print(" ", path)
    raise FileNotFoundError(
        "training/train.py not found. Re-run .\\scripts\\push_kaggle_kernel.ps1 "
        "and confirm agentguard-code.zip is attached under Input."
    )

required = ("scoring.py", "tokenizer_utils.py", "probe_checkpoint.py", "export_onnx.py")
missing = [name for name in required if not (working / "training" / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Training bundle incomplete; missing {missing}. Re-run push_kaggle_kernel.ps1."
    )

# Fail fast if a stale generator-slice diagnostic slipped into the zip.
train_src = (working / "training" / "train.py").read_text(encoding="utf-8")
if "(r for r in train_rows" in train_src:
    raise RuntimeError(
        "Stale training/train.py detected (generator slice bug). "
        "Re-run .\\scripts\\push_kaggle_kernel.ps1 to upload a fresh code dataset."
    )

print("Working tree ready:")
for path in sorted((working / "training").glob("*.py")):
    print(" ", path.relative_to(working))
print(f"PYTHONPATH={os.environ['PYTHONPATH']}")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.copy()
env["PYTHONPATH"] = str(Path("/kaggle/working").resolve())

adv = Path("benchmarks/dataset/adversarial.jsonl")
ben = Path("benchmarks/dataset/benign.jsonl")
print(f"Anthropic adversarial exists: {adv.exists()} ({adv.stat().st_size if adv.exists() else 0} bytes)")
print(f"Anthropic benign exists:      {ben.exists()} ({ben.stat().st_size if ben.exists() else 0} bytes)")
if not adv.exists() or not ben.exists():
    raise FileNotFoundError(
        "Anthropic JSONL missing from the Kaggle code bundle. "
        "On your PC run: .\\scripts\\push_kaggle_kernel.ps1 "
        "(it must upload benchmarks/dataset/*.jsonl)."
    )

subprocess.run(
    [sys.executable, "training/prepare_dataset.py", "--source", "anthropic"],
    check=True,
    env=env,
)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.copy()
env["PYTHONPATH"] = str(Path("/kaggle/working").resolve())

subprocess.run([sys.executable, "training/train.py"], check=True, env=env)
subprocess.run(
    [
        sys.executable,
        "training/probe_checkpoint.py",
        "--hf-only",
        "--data-dir",
        "training/data",
    ],
    check=True,
    env=env,
)
print("HF probe PASS")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.copy()
env["PYTHONPATH"] = str(Path("/kaggle/working").resolve())

subprocess.run(
    [
        sys.executable,
        "training/export_onnx.py",
        "--data-dir",
        "training/data",
    ],
    check=True,
    env=env,
)
subprocess.run(
    [
        sys.executable,
        "training/probe_checkpoint.py",
        "--onnx-only",
        "--data-dir",
        "training/data",
    ],
    check=True,
    env=env,
)
print("ONNX probe PASS - safe to download agentguard/models/")


In [ ]:
from pathlib import Path

print("=== Final Metrics Summary ===")
print("Precision / Recall / F1 / FPR: see training/train.py output above")

onnx_path = Path("agentguard/models/risk_scorer.onnx")
hash_path = Path("agentguard/models/model.sha256")

if not onnx_path.exists():
    raise FileNotFoundError(
        "risk_scorer.onnx was not produced. Check Logs for training/export errors."
    )

print(f"Model SHA-256: {hash_path.read_text().strip()[:16]}...")
print(f"Model size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB")
print("Download agentguard/models/ from the Output tab after this run completes.")

In [ ]:
print(
    "If Output tab lists files under agentguard/models/, download risk_scorer.onnx, model.sha256, and tokenizer files."
)